In [15]:
import sys
from pathlib import Path
from glob import glob
import re
from tbparse import SummaryReader
import pandas as pd

MODEL_DIR = Path("/home/chendawww/workspace/rl-navibot/tries/burger_navi_in_world/saved_models")
TB_LOG_DIR = MODEL_DIR / "SAC/log" # 你存放 TensorBoard 日志的目录
SAVE_FREQ = 4000

print("正在解析 TensorBoard 日志...")
reader = SummaryReader(str(TB_LOG_DIR))
df = reader.scalars

# 🔥 核心指标：只看平均奖励 (如果你想看成功率，把 tag 换成你自定义的 success_rate)
target_tag = "rollout/ep_rew_mean"
metric_df = df[df['tag'] == target_tag].copy()

if metric_df.empty:
    # 有时候 SB3 的 tag 不带 rollout/ 前缀，可能是 ep_rew_mean
    metric_df = df[df['tag'] == "ep_rew_mean"].copy()
    
if metric_df.empty:
    print(f"❌ 未找到 reward 指标，请打开 TensorBoard 确认具体的 Tag 名称！")
    sys.exit(1)

# 🔥 关键操作：滑动平均 (窗口大小设为 5，相当于 5 * SAVE_FREQ = 20000 步的平均表现)
# 这样可以完美过滤掉单次极端好运或极端倒霉的 episode
window_size = 5 
metric_df['smoothed_value'] = metric_df['value'].rolling(window=window_size, min_periods=1).mean()

# 找到平滑后的最大值对应的行
best_row = metric_df.loc[metric_df['smoothed_value'].idxmax()]
best_step = int(best_row['step'])
best_raw_value = best_row['value']
best_smooth_value = best_row['smoothed_value']

print(f"\n=== TensorBoard 分析结果 ===")
print(f"指标: {target_tag}")
print(f"平滑窗口: {window_size * SAVE_FREQ} 步")
print(f"原始最高 Reward: {best_raw_value:.2f}")
print(f"平滑最高 Reward: {best_smooth_value:.2f}")
print(f"🎯 选定的训练步数: {best_step}")

# 拼接模型路径
target_model_path = MODEL_DIR / f"sac_nav_model_{best_step}_steps.zip"

if not target_model_path.exists():
    print(f"\n⚠️ 警告：TB 显示最优步数是 {best_step}，但找不到对应的 zip 文件！")
    print("正在进行兜底查找（找最接近的已保存断点）...")
    
    all_zips = glob(str(MODEL_DIR / "SAC/sac_nav_model_*_steps.zip"))
    steps_exist = []
    for path in all_zips:
        match = re.search(r"sac_nav_model_(\d+)_steps", path)
        if match:
            steps_exist.append((int(match.group(1)), path))
            
    if steps_exist:
        steps_exist.sort(key=lambda x: x[0])
        closest_step, closest_path = min(steps_exist, key=lambda x: abs(x[0] - best_step))
        print(f"🟡 兜底选择最接近的模型: {closest_path} (Step: {closest_step})")
        target_model_path = Path(closest_path)
    else:
        print("❌ 彻底找不到模型文件，退出。")
        sys.exit(1)

print(f"\n✅ 最终确定的迁移学习基座模型:\n   {target_model_path}")
print("\n接下来，拿着这个路径去 house 环境里开搞微调吧！")


正在解析 TensorBoard 日志...

=== TensorBoard 分析结果 ===
指标: rollout/ep_rew_mean
平滑窗口: 20000 步
原始最高 Reward: 344.20
平滑最高 Reward: 344.53
🎯 选定的训练步数: 134135

⚠️ 警告：TB 显示最优步数是 134135，但找不到对应的 zip 文件！
正在进行兜底查找（找最接近的已保存断点）...
🟡 兜底选择最接近的模型: /home/chendawww/workspace/rl-navibot/tries/burger_navi_in_world/saved_models/SAC/sac_nav_model_136000_steps.zip (Step: 136000)

✅ 最终确定的迁移学习基座模型:
   /home/chendawww/workspace/rl-navibot/tries/burger_navi_in_world/saved_models/SAC/sac_nav_model_136000_steps.zip

接下来，拿着这个路径去 house 环境里开搞微调吧！


In [12]:
best_step // 4000

33